
# **This script is for generating test cases and golden datasets.**

My plan for the evaluation workflow is to generate a golden dataset that would contain a question and a golden answer per a context.Agains this dataset I will perform evaluation of my RAG system. I use Qwen model for my RAG system and decided to use a different one for generating the golden dataset: gpt-oss:20b. Generating such dataset is computationally expensive, and I am not using any paid api access. My workaround is to extract firs 50 articles' abstracts for contexts rather than splitting the whole article for chunks. Abstract sizes already contain good lengths and having a variety of chunks is better then if I had used 50 chunks of the same article. Luckly, abstracts are semantically whole and have a perfect length for a chunk on avg = 188.
The context is represented by a list of lists each containing a collection of single words. Dividing the string by sentences, increases the load to the model and it times out.

In [1]:
import csv
import json
import numpy as np
from deepeval.models import OllamaModel
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
import json
import re
from deepeval.models import OllamaModel
from deepeval.synthesizer import Synthesizer
from pygments.lexers.mosel import FUNCTIONS
import pandas as pd
from IPython.display import display, Markdown
from golden_generator import GoldenGenerator

In [2]:
ollama_model = OllamaModel(
    model="gpt-oss:20b",
    base_url="http://localhost:11434",
    temperature=0
)

synthesizer = Synthesizer(
    model=ollama_model,
    async_mode=False
)



In [3]:
golden_gen = GoldenGenerator(model=ollama_model, context_file="parsed_articles.json", synthesizer=synthesizer)
documents = golden_gen.prepare_context(split_by=" ", count=50)

In [4]:
mean_length = np.mean([len(x) for x in documents])

print(f"Mean abstarct length: {mean_length}")

Mean abstarct length: 188.4


<font color="red">KEEP THIS FUNCTION CALLS COMMENTED OUT!</font>

In [9]:
#g = golden_gen.synthesize_save_golden(documents)

<font color="red"> _________________</font>

After I have generated 50 goldens using gpt-oss:20b, I have decided to feed the file to ChatGPT and ask it to evaluate the quality of the generated dataset. ChatGPT decided that 42 % of the questions and answeres were poor quality. I ran over some of them to test them manually, both approved and disapproved entries.

In [5]:
chat_gpt_eval = pd.read_csv("goldens_evaluation.csv")
print(chat_gpt_eval.columns)

Index(['golden_id', 'status', 'reason', 'keep_for_evaluation'], dtype='str')


In [6]:
print(f" The number of poor goldens evaluated by ChatGpt: {len(chat_gpt_eval[chat_gpt_eval["keep_for_evaluation"] == False])}")
print(f" The percentage of poor goldens evaluated by ChatGpt: {len(chat_gpt_eval[chat_gpt_eval["keep_for_evaluation"] == False])/ len(chat_gpt_eval) * 100}%")

 The number of poor goldens evaluated by ChatGpt: 21
 The percentage of poor goldens evaluated by ChatGpt: 42.0%


In [7]:
poor = chat_gpt_eval[chat_gpt_eval["keep_for_evaluation"] == False]
good = chat_gpt_eval[chat_gpt_eval["keep_for_evaluation"] == True]

In [8]:
poor.head(5)

,golden_id,status,reason,keep_for_evaluation
0,1,REVISE,"The answer adds the mechanism ""model knows whe...",False
1,2,REVISE,"There are unsupported details about sampling, ...",False
3,4,REVISE,"Overall good, but the initialization method is...",False
4,5,REVISE,Repair/differentiation/translation are support...,False
7,8,REVISE,Mathematical details/equations are added that ...,False


In [9]:
golden_df = pd.read_csv("golden_dataset.csv")
bad_id = 2
q = golden_df[golden_df["golden_id"] == bad_id]["question"].iloc[0]
a = golden_df[golden_df["golden_id"] == bad_id]["expected_answer"].iloc[0]
c = golden_df[golden_df["golden_id"] == bad_id]["context"].iloc[0].replace("'", "").replace(",", "")
print("Question from a poor entry: ")
display(Markdown(q))
print("Answer from a poor entry: ")
display(Markdown(a))
print("Given context:")
# display(Markdown(c))
print("ChatGPT reasoning:")
# display(Markdown(chat_gpt_eval[chat_gpt_eval["golden_id"] == bad_id]["reason"].iloc[0]))
c

Question from a poor entry: 


How do authors minimize queries, achieve optimal tgt alignment, post‑process, prompting in black‑box generative AI?

Answer from a poor entry: 


**Key ideas from the paper**

| Goal | How it’s achieved |
|------|-------------------|
| **Minimize queries** | • Treat the generator as a black‑box that can be queried repeatedly.<br>• Use a *sampling‑selection* algorithm that, at each step, chooses the next query so that the expected number of queries to reach the target distribution is as small as possible. |
| **Optimal target‑distribution alignment** | • Formulate exact and approximate alignment as a statistical matching problem.<br>• Prove that the proposed algorithms are *query‑optimal* in expectation when the number of requested outputs \(m\) grows large ( \(m\to\infty\) ). |
| **Post‑processing** | • After collecting \(m\) outputs, apply a lightweight post‑processing step that re‑weights or resamples the outputs to match the target attribute distribution more closely. |
| **Prompting** | • Use prompt‑engineering as a complementary front‑end: craft prompts that bias the generator toward the desired distribution, then apply the query‑optimal post‑processing to fine‑tune the result. |

**Result** – The combination of query‑optimal sampling, asymptotically optimal alignment, and lightweight post‑processing (with optional prompting) yields statistically better attribute distributions in black‑box generative AI, as shown in text‑to‑image and geocoded persona experiments.

Given context:
ChatGPT reasoning:


'[Generative AI systems are increasingly used but aligning their outputs with user requirements poses a continuing challenge. Here we aim to ensure that the distribution of an attribute of an AI-generated output aligns with a user-specified target. This is motivated by examples such as fairness where we want to ensure that a protected attribute (e.g. gender race or age categories) follows a desired distribution and synthetic data generation where we want the generated data to be representative of a target distribution. We study the practically important black-box access setting where a user can repeatedly query a generative AI model. The goal is to return $m\\\\ge 1$ outputs whose joint attribute distribution is as close as possible to this target. For both exact and approximate alignment we develop algorithms that minimize the expected number of queries to the generator and we further demonstrate their optimality as the number of requested outputs $m \\\\rightarrow \\\\infty$. Experim

In [10]:
good_id = 3
q = golden_df[golden_df["golden_id"] == good_id]["question"].iloc[0]
a = golden_df[golden_df["golden_id"] == good_id]["expected_answer"].iloc[0]
c = golden_df[golden_df["golden_id"] == good_id]["context"].iloc[0].replace("'", "").replace(",", "")
print("Question from a good entry: ")
display(Markdown(q))
print("Answer from a good entry: ")
display(Markdown(a))
print("Given context:")
display(Markdown(c))
print("ChatGPT reasoning:")
display(Markdown(chat_gpt_eval[chat_gpt_eval["golden_id"] == good_id]["reason"].iloc[0]))

Question from a good entry: 


How does static compact doc alone affect issue resolution in real repos vs. source present?

Answer from a good entry: 


Static compact documentation alone does **not** improve issue resolution. In the experiments, when the source code was available, neither static compact docs nor retrieved context outperformed the agent that was given only the issue description. Thus, static compact docs alone do not help resolve real‑repo issues better than the issue alone.

Given context:


[We investigate whether natural-language documentation helps coding agents resolve software issues and we build the tools to construct and evaluate it. We introduce a roundtrip benchmark that scores code descriptions by whether code regenerated from them passes the original tests and show that completeness not length drives a "descriptions" fidelity. Using the benchmark as an optimization signal we discover a description-writing prompt that reaches full fidelity and generalizes to unseen files. We then test the hypothesis that motivated the work: that better documentation helps an agent resolve real repository issues. Across two model families and ten repositories and against a positive control confirming that our evaluation can detect a genuine improvement we find that it does not. When the source is present neither static compact documentation nor retrieved context beats the issue alone. We report this negative result together with the benchmark and the optimizer and we characterize the boundary at which documentation helps.]

ChatGPT reasoning:


The answer follows directly from the context.

I will feed the bad answers to ChatGPT  and prompt it to substitude with better ones, and then I will replace them in my file. After that I will have a file with golden answers for tests.

In [11]:
poor_csv = golden_df[golden_df["golden_id"].isin(poor["golden_id"])].to_csv("poor.csv", index=False)

In [12]:
revised_df = pd.read_csv("poor_revised_goldens.csv")
golden_df = golden_df.set_index("golden_id")

revised_df = revised_df.set_index("golden_id")

golden_df.update(
    revised_df[["question", "expected_answer"]]
)

golden_df = golden_df.reset_index()
print(f"revised question: {golden_df[golden_df["golden_id"] == bad_id]["question"].iloc[0]}")
print(f"revised answer: {golden_df[golden_df["golden_id"] == bad_id]["expected_answer"].iloc[0]}")

revised question: What approach do the authors propose for aligning black-box generative AI outputs with a user-specified target distribution?
revised answer: The authors develop algorithms for exact and approximate alignment that minimize the expected number of queries to a black-box generative AI model. They show that these algorithms become optimal as the number of requested outputs approaches infinity. Experiments on text-to-image and geocoded persona generation show that the proposed post-processing algorithms improve statistical attribute alignment and complement prompting-based interventions.
